# ಪಾಠ 18: ಕ್ರಿಪ್ಟೋಗ್ರಾಫಿಕ್ ರಿಸೀಟ್‌ಗಳೊಂದಿಗೆ AI ಏಜೆಂಟುಗಳನ್ನು ಸುರಕ್ಷಿತಗೊಳಿಸುವುದು

## ಹಸ್ತಚಾಲಿತ ನೋಟ್‌ಬುಕ್

ಈ ನೋಟ್‌ಬುಕ್ ನಾಲ್ಕು ವ್ಯಾಯಾಮಗಳಿಂದ ಸಾಗುತ್ತದೆ:

1. ಏಜೆಂಟ್ ಟೂಲ್ ಕರೆಗಾಗಿ ನಿಮ್ಮ ಮೊದಲ ರಿಸೀಟ್‌ಗೆ **ಅಂಕುರಿತ ಮಾಡಿ** ಮತ್ತು ಅದನ್ನು ಪರಿಶೀಲಿಸಿ.
2. **ರಿಸೀಟ್‌ನ್ನು ತೊಂದರೆ ಪಡಿಸಿ** ಮತ್ತು ಪರಿಶೀಲನೆ ವಿಫಲವಾಗುವುದನ್ನು ನೋಡಿ.
3. **ಮೂರು-ರಿಸೀಟ್ ಸರಪಳಿಯನ್ನು ನಿರ್ಮಿಸಿ** ಮತ್ತು ಸರಪಳಿ ಅಖಂಡತೆಯನ್ನು ದೃಢೀಕರಿಸಿ.
4. **ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್ ಟೂಲ್ ಕರೆಯನ್ನು ಸಾಗಿಸಿ** ಇದರಿಂದ ಪ್ರತಿಯೊಂದು ಕ್ರಿಯೆಯೂ ಒಂದು ರಿಸೀಟ್ ಅನ್ನು ಉದ್ಘಾಟಿಸುತ್ತದೆ.

ಎಲ್ಲಾ ಕ್ರಿಪ್ಟೋಗ್ರಾಫಿಕ್ ಮೂಲಗಳು ಚೆನ್ನಾಗಿ ನಿರ್ವಹಿಸಲಾದ ಗ್ರಂಥಾಲಯಗಳಿಂದ ಆಮದು ಮಾಡಿಕೊಳ್ಳಲ್ಪಟ್ಟಿವೆ (`pynacl` Ed25519 ಗಾಗಿ, `jcs` RFC 8785 ಕ್ಯಾನನಿಕಲ್ JSON ಗಾಗಿ, `hashlib` Python ಸ್ಟ್ಯಾಂಡರ್ಡ್ ಲೈಬ್ರರಿಯಿಂದ SHA-256 ಗಾಗಿ). ರಿಸೀಟ್ ಲಾಜಿಕ್ ಸ್ವತಃ ಸರಳ Python ಆಗಿದ್ದು, ನೀವು ಓದಿ ತಿದ್ದುಪಡಿ ಮಾಡಬಹುದು.

ಕೋಶಗಳನ್ನು ಕ್ರಮವಾಗಿ ಚಲಾಯಿಸಿ. ಪ್ರತಿ ವಿಭಾಗವು ಸಣ್ಣದು ಮತ್ತು ಸ್ವ-ಸ್ವತಂತ್ರವಾಗಿದೆ.


## ಸೆಟ್‌ಅಪ್

ಎರಡು ಅವಲಂಬನಿಗಳನ್ನು ಇನ್‌ಸ್ಟಾಲ್ ಮಾಡಿ. ಎರಡಕ್ಕೂ ಪರವಾನಗಿ ಅನುಮತಿಗಳು (Apache-2.0 / MIT) ಇವೆ.


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## ಸಹಾಯಕ ಉಪಯುಕ್ತತೆಗಳು

ಈ ಎರಡು ಸಹಾಯಕಗಳು ಬೇಸ್64url ಎನ್‌ಕೋಡಿಂಗ್ (ಪ್ಯಾಡಿಂಗ್ ಇಲ್ಲದೆ) ಮತ್ತು 任意オブジェクトಗಳ SHA-256 ಹ್ಯಾಶಿಂಗ್ ಅನ್ನು ನಿರ್ವಹಿಸುತ್ತವೆ. ಇವುಗಳು ನೋಟ್ಬುಕ್‌ನ ಉಳಿದ ಭಾಗವನ್ನು ಸ್ವೀಕೃತಿ ತರ್ಕದ ಮೇಲೆ ಕೇಂದ್ರೀಕರಿಸಿರುತ್ತವೆ.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## ಅನুচ್ಛೇದ 1: ನಿಮ್ಮ ಮೊದಲ ರಸೀದಿಯನ್ನು ಸಹಿ ಮಾಡಿ

ನಮ್ಮ **Contoso ಪ್ರಯಾಣ** ಏಜೆಂಟ್ ಒಬ್ಬ ಗ್ರಾಹಕನಿಗೆ ಸಿಡ್ನಿ ನಿಂದ ಲಾಸ್ ಏಂಜಲಿಸ್ ಗೆ ಫ್ಲೈಟ್‌ಗಳನ್ನು ಹುಡುಕಿದನು ಎಂದು ಊಹಿಸಿ. ಭವಿಷ್ಯದ ಪರಿಶೀಲಕನು ನಮಗನುಂಬಿಕೆಯಿಲ್ಲದೆ ಇದನ್ನು ಪರಿಶೀಲಿಸ ಬಹುದಾಗಿಸಲು ನಾವು ಈ ಉಪಕರಣ ಕರೆವನ್ನು ಸಹಿ ಮಾಡಲಾದ ರಸೀದಿಯಾಗಿ ದಾಖಲಿಸಲು ಬಯಸುತ್ತೇವೆ.

### ಹಂತ 1.1: ಸಹಿ ಮಾಡುವ ಕೀ ಅನ್ನು ಉತ್ಪತ್ತಿ ಮಾಡಿ

ಉತ್ಪಾದನೆಯಲ್ಲಿ, ಏಜೆಂಟ್ ಅವರ ಸಹಿ ಮಾಡುವ ಕೀ ಹಾರ್ಡ್ವೇರ್ ಸೆಕ್ಯುರಿಟಿ ಮ್ಯಾಡ್ಯೂಲ್ (HSM), ಆಜೂರ್ ಕೀ ವಾಲ್ಟ್, ಅಥವಾ ಹೋಲುವ ಯಾವುದೇ ರಕ್ಷಿತ ಸಂಗ್ರಹಣೆಯಲ್ಲಿ ಇರುತ್ತದೆ. ಈ ಪಾಠಕ್ಕಾಗಿ ನಾವು ಸ್ಮರಣೆಯಲ್ಲಿ ಹೊಸ ಕೀ ಅನ್ನು ಉತ್ಪತ್ತಿ ಮಾಡುತ್ತೇವೆ.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### ಹಂತ 1.2: ರಸೀದಿ ಪೇಲೋಡ್ ನಿರ್ಮಾಣ ಮಾಡು

ಪೇಲೋಡ್‌ನಲ್ಲಿ ನಾವು ರಸೀದಿ ದೃಢೀಕರಿಸಬೇಕಾದ ಎಲ್ಲವೂ ಸಿಕ್ಕಿರುವುದು: ಯಾರು ಕಾರ್ಯನಿರ್ವಹಿಸಿದರು, ಯಾವ ಉಪಕರಣ, ಯಾವ ಆರ್ಗ್ಯುಮೆಂಟ್‌ಗಳು, ಏನು ಮರಳಿತು, ಯಾವ ನೀತಿಯಡಿ ಮತ್ತು ಯಾವಾಗ. ನಾವು ಆರ್ಗ್ಯುಮೆಂಟ್‌ಗಳು ಮತ್ತು ಫಲಿತಾಂಶವನ್ನು ಸದ್ಯದಲ್ಲೇ ಸೇರಿಸುವುದನ್ನು ಬದಲಿಗೆ ಅವುಗಳನ್ನು ಹ್ಯಾಶ್ ಮಾಡುತ್ತೇವೆ, ಇದರಿಂದ ರಸೀದಿ حساس సమాచారం ಹರಡುವುದನ್ನು ತಡೆಯುತ್ತದೆ.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### ಹಂತ 1.3: ಸ್ವೀಕೃತಿಯನ್ನು ಸಹಿ ಮಾಡಿ ಮತ್ತು ಸಂಯೋಜಿಸಿ

ಮೂರು ಹಂತಗಳು:

1. ಜೇಸಿಎಸ್ ಬಳಸಿ ಪೇಲೋಡ್ ಅನ್ನು ಕ್ಯಾನನಿಕಲ್ ಮಾಡಿ, ಹಾಗಾಗಿ ಎರಡು ಜಾರಿಗೆಳಿಸುವಿಕೆಗಳು ಒಂದೇ ತಾರ್ಕಿಕ ಸ್ವೀಕೃತಿಯನ್ನು ಉತ್ಪಾದಿಸಿದಾಗ ಅದೇ ಬೈಟ್ ಸರಣಿಯನ್ನು ಉತ್ಪಾದಿಸಬೇಕು.
2. ಕ್ಯಾನನಿಕಲ್ ಬೈಟ್‌ಗಳನ್ನು SHA-256 ನೊಂದಿಗೆ ಹ್ಯಾಶ್ ಮಾಡಿ.
3. Ed25519 ಖಾಸಗಿ ಕೀ ಬಳಸಿ ಹ್ಯಾಶ್ ಗೆ ಸಹಿ ಮಾಡಿ.

ಸಹಿ ನಂತರ ಮೂಲ ಪೇಲೋಡ್‌ಗೆ ಸಂಯೋಜಿಸಲಾಗುತ್ತದೆ ಮತ್ತು ಅಂತಿಮ ಸ್ವೀಕೃತಿ ನಿರ್ಮಿಸಲಾಗುತ್ತದೆ.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### ಹಂತ 1.4: ರಸೀದಿ ಪರಿಶೀಲಿಸಿ

ಪರಿಶೀಲನೆ ಪ್ರಕ್ರಿಯೆಯನ್ನು ಹಿಂತಿರುಗಿಸುತ್ತದೆ. ನಾವು ಸಹಿಯನ್ನು ತೆಗೆದುಹಾಕುತ್ತೇವೆ, ಕಾನನಿಕಲ್ ಹ್ಯಾಶ್ ಅನ್ನು ಪುನಃ ಗಣನೆ ಮಾಡುತ್ತೇವೆ, ಮತ್ತು ರಸೀದಿಯಲ್ಲಿ ಇರುವ ಸಾರ್ವಜನಿಕ ಕೀ ವಿರುದ್ಧ ಸಹಿಯನ್ನು ಪರಿಶೀಲಿಸುತ್ತೇವೆ.

ಈ ಪರಿಶೀಲನೆಯನ್ನು ಮಾಡುವ ಪರಿಶೋಧಕನಿಗೆ ನಮ್ಮಿಂದ ಬೇರೆ ಯಾವುದೇ ಅವಶ್ಯಕತೆ ಇರುವುದಿಲ್ಲ, ಕೇವಲ ರಸೀದಿಯೇ ಬೇಕು. ಯಾವುದೇ ಸೇವೆಯನ್ನು ಕರೆಮಾಡಬೇಕಾಗಿಲ್ಲ, ಯಾವುದೇ ಕೀ ಡೈರೆಕ್ಟರಿಯನ್ನು ವಿಚಾರಿಸಬೇಕಾಗಿಲ್ಲ, ಯಾವುದೇ ನಂಬಿಕೆ ಬೇಕಾಗಿಲ್ಲ.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

ನೀವು `ರೂಪಾಯಿ ಮಾನ್ಯವಾಗಿದೆ: ಸತ್ಯ` ಎಂದು ನೋಡಬೇಕು. ಏಜೆಂಟ್ ತನ್ನ ಮೊದಲ ಕ್ರಿಪ್ಟೋಗ್ರಾಫಿಕ್ ಆಗಿ ಸಹಿ ಮಾಡಲಾದ ಪರಿಶೀಲನಾ ದಾಖಲೆಯನ್ನು ಉತ್ಪಾದಿಸಿಕೊಂಡಿದೆ.


## ವಿಭಾಗ 2: ರಸೀದೆ ಜೊತೆಗೆ ಮಾಹಿತಿ ಬದಲಿಸುವುದು

ರಸೀದೆಗಳ ಸಂಪೂರ್ಣ ಉದ್ದೇಶವು ಅವು ತಿರುವು-ಸೂಚಕವಾಗಿರುವುದಾಗಿದೆ. ಅದನ್ನು ನಾವು ಸಾಬೀತುಪಡಿಸೋಣ.

ನಾವು ರಸೀದೆನ ಒಂದು ಅಕ್ಷರವನ್ನಷ್ಟೇ ಬದಲಿಸಿ ದೃಢೀಕರಣ ವಿಫಲವಾಗುವುದನ್ನು ನೋಡೋಣ.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### ಏನು ನಡೆದಿತು?

ನಾವು `policy_id` ಅನ್ನು ಬದಲಾಯಿಸಿದಾಗ, ಕ್ಯಾನೋನಾ ಬೈಟ್ಗಳು ಬದಲಾಯಿಸಿತು. ಆ ಬೈಟ್ಸ್‌ಗಳ SHA-256 ಹ್ಯಾಶ್ ಬದಲಾಯಿಸಿತು. (ಮೊದಲಿನ ಹ್ಯಾಶ್ ಮೇಲೆ ಇರುವ) ಸಹಿಯು ಹೊಸ ಹ್ಯಾಶ್‌ಗೆ ಹೊಂದಿಕೊಂಡಿಲ್ಲ. ಪರಿಶೀಲನೆ ಸರಿಯಾಗಿ `False` ಅನ್ನು ಮರಳಿಸುತ್ತದೆ.

ರಿಸೀಟ್‌ನ ಯಾವುದೇ ಕ್ಷೇತ್ರವನ್ನು ಬದಲಾಯಿಸಿ ಪರಿಶೀಲಿಸಬಹುದಾದ ರೀತಿಯಲ್ಲಿ ಇರುವುದಿಲ್ಲ, ಮಾಗುವವರಿಗೆ ಖಾಸಗಿ ಕೀ ಇಲ್ಲದಿದ್ದರೆ. ಖಾಸಗಿ ಕೀ ಒಂದು ಕೀ ವಾಕಲ್ಟ್‌ನಲ್ಲಿ ಇರುತ್ತಿರುವಾಗ ಮತ್ತು ಸಾರ್ವಜನಿಕ ಕೀ ಪ್ರಕಟವಾಗಿರುವಾಗ, ತಿದ್ದುಪಡಿ ಮಾಡಲು ಸಾಧ್ಯವಿಲ್ಲ.

ನೀವು ಸ್ವತಃ ಪ್ರಯತ್ನಿಸಿ: ಮೇಲಿನ ಸೆಲ್‌ನಲ್ಲಿ `tool_name` ಅಥವಾ `agent_id` ಅಥವಾ `timestamp` ಅನ್ನು ಬದಲಾಯಿಸಿ ಮತ್ತೆ ಓಡಿಸಿ. ಪ್ರತಿಯೊಂದು ಬದಲಾವಣೆ ಅನಾಮಾನ್ಯ ರಿಸೀಟ್ ಅನ್ನು ಉತ್ಪಾದಿಸುತ್ತದೆ.


## ವಿಭಾಗ 3: ಚೈನ್ ರಸೀದಿಗಳನ್ನು ಒಂದೊಂದಾಗಿ ಜೋಡಿಸುವುದು

ಒಂದು ರಸೀದಿ ಒಂದೊಂದು ಕ್ರಿಯೆಯನ್ನು ರಕ್ಷಿಸುತ್ತದೆ. ಹೆಚ್ಚಿನ ಏಜೆಂಟ್‌ಗಳು ಅನೇಕ ಕ್ರಿಯೆಗಳು ತೆಗೆದುಕೊಳ್ಳುತ್ತಾರೆ. ಸಂಪೂರ್ಣ ಕ್ರಮವನ್ನು ತಿರುವು ತೋರಿಸುವಂತೆ ಮಾಡಲು, ನಾವು ಹೊಸ ರಸೀದಿಯ ಪೇಲಾಗ್ನ್ನಲ್ಲಿ ಹಳೆಯ ರಸೀದಿಯ ಹ್ಯಾಶ್ ಅನ್ನು ಸೇರಿಸಿಯೇ ಪ್ರತಿಯೊಬ್ಬ ರಸೀದಿಯನ್ನು ಹಿಂದಿನ ರಸೀದಿಗೆ ಲಿಂಕ್ ಮಾಡುತ್ತೇವೆ.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

ಯಾರಾದರೂ ರಸೀದಿಯನ್ನು ತೆಗೆದುಹಾಕಿದರೆ ಅಥವಾ ಮರುಕ್ರಮಿಸುತ್ತಿದ್ದರೆ, ಸರಣಿಯು ಆ ಬಿಂದುವಿನಲ್ಲಿ ಮುರಿಯುತ್ತದೆ. ಯಾವುದೇ ನಂತರದ ರಸೀದಿಯ ಪರಿಶೀಲನೆ ವಿಫಲವಾಗುತ್ತದೆ ಏಕೆಂದರೆ ಅದರ `previous_receipt_hash` ಈಗ ಅದರ ಹಳೆಯದಿನ ಹ್ಯಾಶ್ ಗೆ ಹೊಂದಿಕೆಯಾಗುವುದಿಲ್ಲ.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

ಮಧ್ಯದ ರಸಿದನ್ನು ತಿಪ್ಪಣಿಗೆ ಒಳಪಡಿಸಿ ಸರಪಳಿಯನ್ನು ಮುರಿಯಿರಿ ಮತ್ತು ಮರು-ಪರಿಶೀಲಿಸಿ. ತಿಪ್ಪಣಿಗೊಳಗೊಂಡ ರಸಿದವು ತನ್ನ ಸಹಿ ಪರಿಶೀಲನೆಯಲ್ಲಿ ವಿಫಲವಾಗುತ್ತದೆ, ಮತ್ತು ಮುಂದಿನ ರಸಿದವು ತನ್ನ ಸರಪಳಿ-ಕೊಂಡಿ ಪರಿಶೀಲನೆಯಲ್ಲಿ ವಿಫಲವಾಗುತ್ತದೆ (ಕಾರಣ ಅದರ `previous_receipt_hash` ಈಗ ತಿದ್ದುಪಡಿಯಾದ ಮಧ್ಯದ ರಸಿದದ ಹ್ಯಾಶ್ ಗೆ ಹೊಂದಿಕೆಯಾಗುವುದಿಲ್ಲ).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

ರಸೀದಿ 0 ಇನ್ನೂ ಪರಿಶೀಲನೆ ನಡೆಸುತ್ತದೆ (ಅದು ಬದಲಾಯಿಸಲಾಗಿಲ್ಲ ಮತ್ತು ಅವಲಂಬಿಸಬೇಕಾದ ಪೂರ್ವೋತ್ಪನ್ನ ಇರಲಿಲ್ಲ). ರಸೀದಿ 1 ತನ್ನ ಸಹಿ ಪರಿಶೀಲನೆಯಲ್ಲಿ ವೈಫಲ್ಯಗೊಳ್ಳುತ್ತದೆ ಏಕೆಂದರೆ ನಾವು `tool_args_hash` ಅನ್ನು ಬದಲಾಯಿಸಿದ್ದೇವೆ. ರಸೀದಿ 2 ತನ್ನ ಸರಪಳಿ-ಲಿಂಕ್ ಪರಿಶೀಲನೆಯಲ್ಲಿ ವೈಫಲ್ಯಗೊಳ್ಳುತ್ತದೆ ಏಕೆಂದರೆ ಅದರ `previous_receipt_hash` ಮೂಲ (ಈಗ ಬದಲಾಯಿಸಲಾದ) ರಸೀದಿ 1 ಮೇಲೆ ಲೆಕ್ಕಿಸಲಾಯಿತು.

ಒಡೆಯಗಾರ ಬದಲಾಯಿಸಲಾದ ರಸೀದಿ 1 ಅನ್ನು ಮರು-ಹಸ್ತಾಕ್ಷರ ಮಾಡಿದರೂ (ಆವರಿಗೆ ಖಾಸಗಿ ಕೀ ಇಲ್ಲದೆ ಅದು ಸಾಧ್ಯವಿಲ್ಲ), ರಸೀದಿ 2 ರಲ್ಲಿ ಸರಪಳಿ-ಲಿಂಕ್ ಅಸಮ್ಮತಿಕತೆಯು ಹಿಮ್ಮುಖತೆಯನ್ನು ಬಹಿರಂಗಪಡಿಸುತ್ತದೆ. ಬದಲಾವಣೆಯನ್ನು ಮರೆಮಾಚಲು, ಒಡೆಯಗಾರ ಬದಲಾವಣೆಯ ಬಿಂದುವಿನಿಂದ ಪ್ರಾರಂಭಿಸಿ ಪ್ರತಿ ರಸೀದಿಯನ್ನು ಮರು-ಹಸ್ತಾಕ್ಷರಿಸಬೇಕಾಗುತ್ತದೆ, ಇದು ಖಾಸಗಿ ಕೀ ಹೊಂದಿರುವುದನ್ನು ಅಗತ್ಯವಿರುತ್ತದೆ.


## ವಿಭಾಗ 4: ರਸੀದಿ ಸೈನ್ ಮಾಡುವ ಮೂಲಕ ಏಜೆಂಟ್ ಉಪಕರಣ ಕರೆಮಾಡುವುದು

ವಾಸ್ತವಿಕ ನಿಯೋಜನೆಯಲ್ಲಿ, ಪ್ರತಿ ಏಜೆಂಟ್ ರಚನಾಕಾರನು `make_receipt` ಅನ್ನು ಕರೆ ಮಾಡಲು ನೆನಪಿಸಿಕೊಳ್ಳಬೇಕಾಗಿರುವ ಅವಶ್ಯಕತೆ ನಿಮಗೆ ಇರಲ್ಲ. ಪ್ರತಿಯೊಂದು ಉಪಕರಣ ಕಾಲ್‌ಗೆ ರಸೀದಿ ಸಹಿ ಮಾಡುವುದು ಸ್ವಯಂಚಾಲಿತವಾಗಿರಬೇಕೆಂದು ನೀವು ಬಯಸುತ್ತೀರಿ.

ಇದು ಅತ್ಯಂತ ಸರಳ ಮಾದರಿ: ಯಾವುದೇ ಕರೆಮಾಡಬಹುದಾದ ಉಪಕರಣ ಫಂಕ್ಷನ್ ಹೊಂದುವ ಒಂದು ರ್ಯಾಪರ್ ವರ್ಗ, ಇದು ಅದರ ರಸೀದಿಯನ್ನು ಉತ್ಪಾದಿಸುವ ಆವೃತ್ತಿಯನ್ನು ಹಿಂದಿರುಗಿಸುತ್ತದೆ. ಇದು ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್‌ವರ್ಕ್ (`agent_framework.azure`) ಸೇರಿದಂತೆ ಯಾವುದೇ ಏಜೆಂಟ್ ಫ್ರೇಮ್‌ವರ್ಕ್ಗೆ ಹೊಂದಿಕೊಳ್ಳಲು ಸಾಧ್ಯ.

ನೀವು Azure AI Foundry ಪ್ರಾಜೆಕ್ಟ್ ಹೊಂದಿದ್ದರೆ ಇಲ್ಲದಿದ್ದರೂ, ಕೆಳಗಿನ ಸ್ಥಳೀಯ ನಕಲು ಮಾದರಿ ಈ ಮಾದರಿಯನ್ನು ತೋರಿಸುತ್ತದೆ.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್‌ನೊಂದಿಗೆ ಸಂಯೋಜನೆ

ಮೇಲಿನ `ReceiptedTool` ಜಾಕೆಟ್ ಫ್ರೇಮ್ವರ್ಕ್-ಸ್ವತಂತ್ರವಾಗಿದೆ. ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್‌ನೊಂದಿಗೆ ನಿರ್ಮಿತ ಏಜೆಂಟ್ ಒಳಗೆ ಇದನ್ನು ಬಳಸಲು, ಜಾಕೆಟ್ ಮಾಡಿದ ಫಂಕ್ಷನ್ ಅನ್ನು ಸಾಧನವಾಗಿ ನೋಂದಣಿ ಮಾಡಿರಿ. ಒಂದು ರೂಢಿಸಂಭ್ರಮ (ನೀವು(mock) ಒಂದು ನಿಜವಾದ Azure AI Foundry ಸಾಧನ ನೋಂದಣಿಯಿಂದ ಬದಲಾಯಿಸುವಿರಿ):

```python
# ಸಂಯೋಜನೆ ರೂಪವನ್ನು ತೋರಿಸುವ ಪ್ರಶ್ನಾಕೋಡ್.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     directions="ನೀವು ಕಂಟೋಸೋ ಟ್ರಾವೆಲ್ ಏಜೆಂಟ್ ...",
#     tools=[receipted_lookup],   # ಸುತ್ತಿಕೊಂಡಿರುವ ಸಾಧನ, ಕಚ್ಚಾ ಫಂಕ್ಷನ್ ಅಲ್ಲ
# )
# response = agent.run("ಜೂನ್‌ನಲ್ಲಿ ಸಿಡ್ನಿಯಿಂದ ಲಾಸ್ ಏಂಜಲಿಸ್‌ಗೆ ಪ್ರಚಾರಗಳನ್ನು ಹುಡುಕಿ.")
#
# # ಓಟದ ನಂತರ, ಏಜೆಂಟ್ ಮಾಡಿದ ಪ್ರತಿಯೊಂದು ಸಾಧನ ಕರೆಗೂ ಸಹಿ ಮಾಡಲಾದ ರಸೀದಿ ಇರುತ್ತದೆ:
# audit_chain = receipted_lookup.receipts
```

ಎಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್‌ಗೆ ರಸೀದುಗಳ ಬಗ್ಗೆ ಯಾವುದೇ ಮಾಹಿತಿ ಬೇಕಾಗಿಲ್ಲ. ರಸೀದು ಸಹಿ ಸಾಧನದ ಸುತ್ತಲೂ ಜಾಕೆಟ್ ಮಾಡಲಾಗಿದೆ, ಫ್ರೇಮ್ವರ್ಕ್‌ಗೆ ದೃಢವಾಗಿ ಅಂಟಿಸಲಾಗಿದೆ ಎಂಬುದು ಇಲ್ಲ. ಇದು ನೀವು ಏಜೆಂಟ್ ನವೀಕರಿಸುವ ಅಗತ್ಯವಿಲ್ಲದೆ ಈಗಿರುವ ಏಜೆಂಟ್ ಕೋಡ್‌ಗೆ ಮೂಲವನ್ನು ಸೇರಿಸುವ ವಿಧಾನವಾಗಿದೆ.


## ಪುನಃಸ್ಮರಣೆ ಮತ್ತು ವಿಸ್ತಾರ ಚಾಲೆಂಜ್

ನೀವು:

- Ed25519 ಕೀ ಜೋಡಿ ರಚಿಸಲಾಗಿದೆ.
- ಏಜೆಂಟ್ ಟೂಲ್ ಕರೆಗಾಗಿ ರಸೀದಿಯನ್ನು ರಚಿಸಿ ಸಹಿ ಮಾಡಲಾಗಿದೆ.
- ಸಾರ್ವಜನಿಕ ಕೀ ಬಳಸಿ ಆಫ್‌ಲೈನ್‌ನಲ್ಲಿ ರಸೀದಿಯನ್ನು ಪರಿಶೀಲಿಸಲಾಗಿದೆ.
- ರಸೀದಿಯನ್ನು ಹೇರಳಿಸಿ ಪರಿಶೀಲನೆ ವಿಫಲವಾಗಿರುವುದನ್ನು ಗಮನಿಸಲಾಗಿದೆ.
- ಮೂರು ರಸೀದಿಗಳ ಹ್ಯಾಶ್-ಚೈನ್ ಸರಣಿಯನ್ನು ನಿರ್ಮಿಸಲಾಗಿದೆ.
- ಸರಣಿಯ ಮಧ್ಯಭಾಗವನ್ನು ಹೇರಳಿಸಿ ಸಹಿ ವಿಫಲತೆ ಮತ್ತು ಸರಣಿ ಸಂಪರ್ಕ ವಿಫಲತೆಯನ್ನು ಎರಡನ್ನೂ ಗಮನಿಸಲಾಗಿದೆ.
- ಸ್ವಯಂಚಾಲಿತ ರಸೀದಿ ಸಹಿ ಸಹಿತ ಟೂಲ್ಫಂಕ್ಷನ್ ಅನ್ನು ರೂಢಿಸಲಾಗಿದೆ.

**ವಿಸ್ತಾರ ಚಾಲೆಂಜ್.** ರಸೀದಿ ಸ್ಕೀಮಾದೊಂದಿಗೆ `request_id` ಕ್ಷೇತ್ರವನ್ನು ವಿಸ್ತರಿಸಿ (ವಿತರಣಾ ಟ್ರೇಸಿಂಗ್‌ಗಾಗಿ UUID). ಅದನ್ನು ಸೇರಿಸಲು `make_receipt` ನವೀಕರಿಸಿ, ಮತ್ತು ರಸೀದಿಗಳು ಪೂರ್ಣವಾಗಿ ಪರಿಶೀಲಿಸಲಾಗುತ್ತವೆ ಎಂದು ದೃಢೀಕರಿಸಿ. ನಂತರ ಸಹಿ ಮಾಡುವ ನಂತರ ಕ್ಷೇತ್ರವನ್ನು ಪರಿಷ್ಕರಿಸಿ ಪರಿಶೀಲನೆ ವಿಫಲವಾಗುತ್ತದೆ ಎಂದು ದೃಢೀಕರಿಸಿ. ಇದು ಪದಲ್ ಎನ್‌ಕೋಡಿಂಗ್‌ನ ಪ್ರತಿಯೊಂದು ಬೈಟ್ ಸಹಿಗೆ ಹೇಗೆ ಕೊಡುಗೆ ನೀಡುತ್ತದೆ ಎಂಬುದನ್ನು ನೀವು ಒಳಗೊಳ್ಳಲು ಬಲವಂತ್ರಿಸುತ್ತದೆ.

**ಮುಖ್ಯ ಸೀಮೆ.** ರಸೀದಿಗಳು ಮೂರು ವಿಷಯಗಳನ್ನು ಮಾತ್ರ ಪ್ರಮಾಣೀಕರಿಸುತ್ತವೆ ಮತ್ತು ಆ ಮೂರು ವಿಷಯಗಳೇ: ಹೊಂದಾಣಿಕೆ (ಈ ಕೀ ಈ ವಿಷಯವನ್ನು ಸಹಿ ಮಾಡಿದೆ), ಅಖಂಡತೆ (ಸಹಿ ಮಾಡಿದ ನಂತರ ವಿಷಯ ಬದಲಾಗಿಲ್ಲ), ಮತ್ತು ಕ್ರಮ (ಈ ರಸೀದಿ ಆ ರಸೀದಿಗೆ ನಂತರ ಬಂದಿದೆ). ಅವು ಆ ಏಜೆಂಟ್‌ನ ಕ್ರಿಯೆ ಸರಿಯಾಗಿತ್ತು ಎಂಬುದನ್ನು, `policy_id` ನಲ್ಲಿ ಹೆಸರಿಸಿದ ಧೋರಣೆಯನ್ನು ನಿಜವಾಗಿಯೂ ಮೌಲ್ಯಮಾಪನ ಮಾಡಲಾಗಿದೆಯೇ ಅಥವಾ ಏಜೆಂಟ್ ಪ್ರತಿಯೊಂದು ನಿಯಮವನ್ನು ಪಾಲಿಸಿತಾನೆಯೇ ಎಂಬುದನ್ನು ಪ್ರಮಾಣೀಕರಿಸುವುದಿಲ್ಲ. ರಸೀದಿಗಳು ಆಧಾರದ ಮೇಲೆ ಸ್ಥಾಪನೆಯಾಗಿವೆ. ಆಡಳಿತವು ನೀವು ನಿರ್ಮಿಸುವ ವ್ಯವಸ್ಥೆಯಾಗಿದೆ.

ಅದು ಸೀಮೆಗೆ ಮನಸ್ಸುಮಾಡಿಕೊಂಡು ಪಾಠದ README ಅನ್ನು ಮತ್ತೆ ಓದಿಕೊಳ್ಳಿ. ರಸೀದಿಗಳೊಂದಿಗೆ ತಂಡಗಳು ಮಾಡುವ ಸಾಮಾನ್ಯ ಭ್ರಮೆ "ನಮ್ಮ ಬಳಿ ರಸೀದಿಗಳು ಇರುವುದರಿಂದ" ಅನ್ನೋದು "ನಾವು ಆಡಳಿತ ಹೊಂದಿದ್ದೇವೆ" ಎಂದು ತಿಳಿದುಕೊಳ್ಳುವುದು. ಅದು ಅಲ್ಲ. ರಸೀದಿಗಳು ಏಜೆಂಟ್ ವರ್ತನೆ ಗುರುತಿಸಲು ಸಹಾಯ ಮಾಡುತ್ತವೆ. ಅವು ಅದು ಸರಿಯಾಗಿತ್ತೆಂದು ಸಾಧಿಸುವುದಿಲ್ಲ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
